# Fantasy Premier League (FPL) Team Picker with SQL

This notebook demonstrates how to use SQL to analyse Fantasy Premier League data and select an optimal squad.

**Budget:** £100m  
**Squad:** 15 players (2 GK, 5 DEF, 5 MID, 3 FWD)

## Setup

Install [JupySQL](https://jupysql.ploomber.io/) and connect to an in-memory SQLite database.

In [ ]:
# Install jupysql (runs via micropip in JupyterLite / Pyodide)
import micropip
await micropip.install('jupysql')

In [ ]:
%load_ext sql
%sql sqlite://

## Create the players table

Columns:
- `name` — player name
- `team` — Premier League club
- `position` — GK / DEF / MID / FWD
- `price` — cost in £m
- `total_points` — FPL points scored this season
- `form` — average points per game over the last 4 gameweeks

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS players (
    id         INTEGER PRIMARY KEY,
    name       TEXT    NOT NULL,
    team       TEXT    NOT NULL,
    position   TEXT    NOT NULL CHECK(position IN ('GK','DEF','MID','FWD')),
    price      REAL    NOT NULL,
    total_points INTEGER NOT NULL,
    form       REAL    NOT NULL
);

In [ ]:
%%sql
INSERT OR IGNORE INTO players (id, name, team, position, price, total_points, form) VALUES
-- Goalkeepers
(1,  'Alisson Becker',      'Liverpool',   'GK',  5.5, 140, 7.0),
(2,  'David Raya',          'Arsenal',     'GK',  5.5, 135, 6.5),
(3,  'Nick Pope',           'Newcastle',   'GK',  5.0, 120, 5.5),
(4,  'Matz Sels',           'Nottm Forest','GK',  4.5, 110, 5.0),
-- Defenders
(5,  'Trent Alexander-Arnold','Liverpool', 'DEF', 7.5, 165, 8.0),
(6,  'Virgil van Dijk',     'Liverpool',   'DEF', 6.5, 148, 7.5),
(7,  'Pedro Porro',         'Spurs',       'DEF', 5.5, 130, 6.0),
(8,  'Antonee Robinson',    'Fulham',      'DEF', 5.0, 118, 5.5),
(9,  'Ezri Konsa',          'Aston Villa', 'DEF', 5.0, 115, 5.0),
(10, 'Ben White',           'Arsenal',     'DEF', 6.0, 125, 6.0),
(11, 'Fabian Schar',        'Newcastle',   'DEF', 5.5, 112, 5.2),
-- Midfielders
(12, 'Mohamed Salah',       'Liverpool',   'MID',13.0, 250,12.0),
(13, 'Cole Palmer',         'Chelsea',     'MID',10.5, 200, 9.5),
(14, 'Phil Foden',          'Man City',    'MID', 9.0, 175, 8.5),
(15, 'Bukayo Saka',         'Arsenal',     'MID', 9.5, 185, 9.0),
(16, 'Andreas Pereira',     'Fulham',      'MID', 5.5, 130, 6.0),
(17, 'Emile Smith Rowe',    'Fulham',      'MID', 5.5, 120, 5.5),
(18, 'Bryan Mbeumo',        'Brentford',   'MID', 7.5, 160, 7.5),
-- Forwards
(19, 'Erling Haaland',      'Man City',    'FWD',14.0, 230,11.0),
(20, 'Alexander Isak',      'Newcastle',   'FWD', 8.5, 170, 8.0),
(21, 'Ollie Watkins',       'Aston Villa', 'FWD', 8.5, 165, 7.5),
(22, 'Danny Welbeck',       'Brighton',    'FWD', 6.0, 130, 6.0),
(23, 'Yoane Wissa',         'Brentford',   'FWD', 6.5, 140, 6.5);

## Explore the data

In [ ]:
%%sql
-- All players sorted by total points
SELECT name, team, position, price, total_points, form
FROM   players
ORDER  BY total_points DESC;

In [ ]:
%%sql
-- Best value players (points per £m)
SELECT name, team, position,
       price,
       total_points,
       ROUND(CAST(total_points AS REAL) / price, 1) AS points_per_million
FROM   players
ORDER  BY points_per_million DESC
LIMIT  10;

## Pick an optimal 15-man squad

Rules:
- 2 GK, 5 DEF, 5 MID, 3 FWD
- Max 3 players from any one club
- Total budget ≤ £100m

The query below selects the highest-scoring players per position while respecting the per-club limit and budget.

In [ ]:
%%sql
-- Rank players within each position by total points,
-- then pick the required number per position
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY position
               ORDER BY total_points DESC
           ) AS pos_rank
    FROM players
),
squad AS (
    SELECT * FROM ranked
    WHERE  (position = 'GK'  AND pos_rank <= 2)
        OR (position = 'DEF' AND pos_rank <= 5)
        OR (position = 'MID' AND pos_rank <= 5)
        OR (position = 'FWD' AND pos_rank <= 3)
)
SELECT name, team, position, price, total_points, form,
       SUM(price) OVER () AS squad_cost
FROM   squad
ORDER  BY CASE position
              WHEN 'GK'  THEN 1
              WHEN 'DEF' THEN 2
              WHEN 'MID' THEN 3
              WHEN 'FWD' THEN 4
          END,
          total_points DESC;

## Top player by position

In [ ]:
%%sql
SELECT position,
       name,
       team,
       price,
       total_points
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY position
               ORDER BY total_points DESC
           ) AS rn
    FROM players
)
WHERE rn = 1
ORDER BY CASE position
             WHEN 'GK'  THEN 1
             WHEN 'DEF' THEN 2
             WHEN 'MID' THEN 3
             WHEN 'FWD' THEN 4
         END;

## Points by team

In [ ]:
%%sql
SELECT team,
       COUNT(*)            AS num_players,
       SUM(total_points)   AS team_total_points,
       ROUND(AVG(price),1) AS avg_price
FROM   players
GROUP  BY team
ORDER  BY team_total_points DESC;